# Truck–semitrailer dynamics library

Небольшая C++17-библиотека для дискретной симуляции модели из
`dmpc_model_report.ipynb`. Основной контракт для будущего MPPI:

```cpp
next_state = model.step(state, control, dt);
```

Состояние содержит 9 компонент:

`[X, Y, psi_tractor, psi_trailer, vx, vy, r_tractor, r_trailer, steering_angle]`.

Управление содержит 2 компоненты:

`[steering_rate, longitudinal_acceleration]`.

Вторая компонента нужна, потому что силы в отчёте зависят от `a_cmd`. Если
скоростью управляет внешний PI-регулятор, его выход передаётся именно сюда.

## Сборка и запуск

```bash
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j
ctest --test-dir build --output-on-failure
./build/simulate_truck > trajectory.csv
```

Библиотека не имеет внешних зависимостей. По умолчанию используется RK4;
доступен также явный Euler. `rollout()` принимает заранее выделенные массивы и
не выделяет память — это удобно для тысяч параллельных траекторий MPPI.

## Минимальное использование

```cpp
#include <truck_model/truck_model.hpp>

truck_model::Parameters p;
// Обязательно заменить примерные значения параметрами конкретного грузовика.
p.tractor_mass = 8000.0;
p.trailer_mass = 20000.0;

truck_model::TruckModel model(p);
truck_model::State x;
x.vx = 15.0;
truck_model::Control u{0.1, 0.0};
x = model.step(x, u, 0.01);
```

Для горизонта MPPI:

```cpp
std::vector<truck_model::Control> u(horizon);
std::vector<truck_model::State> x(horizon + 1);
model.rollout(initial, u.data(), horizon, dt, x.data());
```

## Что реализовано

- матрица инерции `M(eta)` и скоростной вектор `k` из отчёта;
- углы увода всех трёх осей и линейная модель боковой силы;
- распределение тяги на заднюю ось и торможения пропорционально нормальным
  нагрузкам;
- общий friction circle для продольной и поперечной силы каждой оси;
- ограничения руля, скорости руля, ускорения и торможения;
- диагностика сил, углов увода и ускорений через `evaluate()`;
- Euler, RK4 и allocation-free rollout.

## Важные границы модели

1. Числа в `Parameters` — только разумные стартовые значения. Для результатов
   диплома нужны масса, инерции, геометрия, нагрузки, жёсткости шин и `mu`
   конкретного автомобиля.
2. В исходном отчёте символ `N_i` используется и как число шин, и как нормальная
   нагрузка. В API это исправлено: `*_tire_count` и `*_normal_load` — разные
   величины.
3. Линейная модель шин с friction circle подходит как первая MPPI-модель, но не
   воспроизводит насыщение Pacejka и перенос вертикальной нагрузки.
4. Регуляризация углов увода ниже `minimum_slip_speed` предназначена для
   численной устойчивости. Текущая постановка ориентирована на движение вперёд,
   а не на точную динамику задним ходом.
5. Для выхода из заноса одной рулевой команды может быть недостаточно. Поэтому
   продольное ускорение оставлено частью управления; при необходимости его можно
   фиксировать или отдавать внешнему регулятору.

## Следующий слой MPPI

Поверх библиотеки понадобятся: функция стоимости (ошибка пути/курса, угол
складывания, боковая скорость, yaw rate, управляющее усилие), генератор шумов,
пакет rollout-траекторий, softmax-веса и receding-horizon цикл. Для начала стоит
проверить модель на стационарном повороте и на сценарии заноса по измеренным или
эталонным данным, а уже затем подбирать веса MPPI.

